# Micro-Sensitivity Explorer

Interaktive Visualisierung der Ergebnisse aus `run_micro_sensitivity.py` (Calibration 7).

| Abschnitt | Inhalt |
|---|---|
| **1 – Übersicht** | Welche Parameter bewegen `resistant_fraction` am stärksten? (Balkendiagramm) |
| **2 – Heatmap** | Normalisierter Einfluss über alle vier Metriken und beide Szenarien |
| **3 – Trajektorien** | Tagesverlauf aller Metriken für einen gewählten Parameter |
| **4 – Wert × Zeit** | `resistant_fraction` als Funktion von Parameterwert und Tag |
| **5 – Szenarien-Vergleich** | kein ABX vs. beta-Lactam: wer gewinnt? |

Passe `OUTPUT_DIR_NAME` und `SELECTED_PARAM` in der Konfigurations-Zelle an.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.colors as pc
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === Konfiguration ===
OUTPUT_DIR_NAME = None          # None → neusten _MicroSensitivity-Output laden
SELECTED_PARAM  = "selection_strength"  # Startwert für Abschnitte 3–5

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
_SENS_RE = re.compile(r"^\d{8}_\d{6}_MicroSensitivity$")


def _resolve_dir(name=None):
    if name:
        p = OUTPUTS_DIR / name
        if not p.exists():
            raise FileNotFoundError(f"Nicht gefunden: {p}")
        return p
    hits = sorted(
        d for d in OUTPUTS_DIR.iterdir() if d.is_dir() and _SENS_RE.match(d.name)
    )
    if not hits:
        raise FileNotFoundError("Kein _MicroSensitivity-Output in outputs/ gefunden.")
    return hits[-1]


OUTPUT_DIR = _resolve_dir(OUTPUT_DIR_NAME)
DATA_DIR   = OUTPUT_DIR / "data"
print("Output:", OUTPUT_DIR.name)

In [ ]:
traj    = pd.read_parquet(DATA_DIR / "sensitivity_trajectories.parquet")
summary = pd.read_parquet(DATA_DIR / "sensitivity_summary.parquet")

try:
    from mss.cli.run_micro_sensitivity import _SWEEP_PARAMS
    PARAM_LABELS = {p.field: p.label for p in _SWEEP_PARAMS}
    PARAM_LOG    = {p.field: p.log_scale for p in _SWEEP_PARAMS}
    PARAM_ORDER  = [p.field for p in _SWEEP_PARAMS]
except ImportError:
    print("⚠  mss nicht importierbar – Feldnamen werden als Labels verwendet.")
    PARAM_LABELS, PARAM_LOG, PARAM_ORDER = {}, {}, []

PARAMS    = [p for p in PARAM_ORDER if p in traj["param"].values] \
            or traj["param"].unique().tolist()
SCENARIOS = traj["scenario"].unique().tolist()
N_DAYS    = int(traj["day"].max())
N_SEEDS   = traj["seed"].nunique()


def lbl(field: str) -> str:
    return PARAM_LABELS.get(field, field)


SCENARIO_COLORS = {"no_abx": "#4c78a8", "beta_lactam": "#f58518"}
SCENARIO_NAMES  = {"no_abx": "kein ABX", "beta_lactam": "beta-Lactam"}
METRIC_LABELS   = {
    "resistant_fraction": "Resistant fraction",
    "total_population":   "Total population",
    "p_clearance":        "p_clearance / day",
    "n_strains":          "Active strains",
}
_PLASMA = px.colors.sequential.Plasma


def _value_colors(values):
    n = len(values)
    return [_PLASMA[int(i / max(n - 1, 1) * (len(_PLASMA) - 1))] for i in range(n)]


def _rgba(hex_color: str, alpha: float = 0.15) -> str:
    r, g, b = pc.hex_to_rgb(hex_color)
    return f"rgba({r},{g},{b},{alpha})"


print(f"Parameter : {len(PARAMS)}")
print(f"Szenarien : {SCENARIOS}")
print(f"Tage      : {N_DAYS}   Seeds: {N_SEEDS}")
print(f"Trajektorien  : {len(traj):>10,} Zeilen")
print(f"Summary       : {len(summary):>10,} Zeilen")

## 1 · Sensitivitäts-Übersicht

Spannweite `max(mean) − min(mean)` von `resistant_fraction` am **letzten simulierten Tag** über alle Sweep-Werte.
Ein langer Balken bedeutet: dieser Parameter verändert das Ergebnis stark.
Sortiert vom einflussreichsten (unten) zum unbedeutendsten (oben).

In [ ]:
_rf = summary[summary["metric"] == "resistant_fraction"].copy()
_rf["label"] = _rf["param"].map(lbl)

_span = (
    _rf.groupby(["param", "scenario", "label"])["mean"]
    .agg(lambda x: x.max() - x.min())
    .reset_index()
    .rename(columns={"mean": "span"})
)
_order = (
    _span.groupby("param")["span"].max()
    .sort_values().index.tolist()
)
_span["label"] = pd.Categorical(
    _span["label"], [lbl(p) for p in _order], ordered=True
)

fig = px.bar(
    _span, y="label", x="span", color="scenario", barmode="group",
    orientation="h",
    color_discrete_map=SCENARIO_COLORS,
    labels={
        "span": "Spannweite resistant_fraction (Endtag)",
        "label": "",
        "scenario": "Szenario",
    },
    title="Sensitivitäts-Übersicht: Einfluss auf resistant_fraction",
)
fig.add_vline(
    x=0.05, line_dash="dot", line_color="gray",
    annotation_text="5 %", annotation_position="top right",
)
fig.update_layout(
    height=max(500, len(PARAMS) * 26 + 130),
    legend_title="Szenario",
    xaxis_range=[0, 1],
)
fig.show()

## 2 · Sensitivitätsheatmap (alle Metriken)

Normalisierte Spannweite pro Metrik: **1.0** = der Parameter mit dem grössten Einfluss auf diese Metrik.
Zeigt, ob ein Parameter nur `resistant_fraction` trifft oder auch `p_clearance`,
Populationsgrösse und Stammvielfalt beeinflusst.

In [ ]:
_rows = []
for _metric in METRIC_LABELS:
    _m = summary[summary["metric"] == _metric]
    _s = (
        _m.groupby(["param", "scenario"])["mean"]
        .agg(lambda x: x.max() - x.min())
        .reset_index()
        .rename(columns={"mean": "span"})
    )
    _s["metric"] = _metric
    _rows.append(_s)
_all = pd.concat(_rows)

for _metric in METRIC_LABELS:
    for _sc in SCENARIOS:
        _mask = (_all["metric"] == _metric) & (_all["scenario"] == _sc)
        _mx = _all.loc[_mask, "span"].max()
        _all.loc[_mask, "rel_span"] = _all.loc[_mask, "span"] / _mx if _mx > 0 else 0.0

for _sc in SCENARIOS:
    _sc_df = _all[_all["scenario"] == _sc]
    _piv   = _sc_df.pivot(index="param", columns="metric", values="rel_span")
    _piv   = _piv.reindex(index=[p for p in reversed(_order) if p in _piv.index])

    fig = go.Figure(go.Heatmap(
        z=_piv.values,
        x=[METRIC_LABELS[c] for c in _piv.columns],
        y=[lbl(p) for p in _piv.index],
        colorscale="YlOrRd",
        zmin=0, zmax=1,
        colorbar=dict(title="rel. Spannweite"),
        hovertemplate="%{y}<br>%{x}: %{z:.2f}<extra></extra>",
    ))
    fig.update_layout(
        title=f"Normalisierte Sensitivität – {SCENARIO_NAMES.get(_sc, _sc)}",
        height=max(420, len(PARAMS) * 24 + 150),
        width=620,
    )
    fig.show()

## 3 · Trajektorien-Explorer

Ändere `SELECTED_PARAM` unten und führe die Zelle neu aus.
Jede Linie entspricht einem Sweep-Wert; Bänder zeigen ±1 Standardabweichung über Seeds.
Zwei Spalten: kein ABX (links) und beta-Lactam (rechts).

In [ ]:
# Verfügbare Parameter:
for _p in PARAMS:
    print(f"  {_p:<44} {lbl(_p)}")

In [ ]:
SELECTED_PARAM = "base_mutation_rate"  # ← hier ändern


def trajectory_fig(param_field: str) -> go.Figure:
    sub    = traj[traj["param"] == param_field].copy()
    values = sorted(sub["param_value"].unique())
    colors = _value_colors(values)
    days   = sorted(sub["day"].unique())
    title  = lbl(param_field)

    metric_info = [
        ("resistant_fraction", "Resistant fraction", False, [0, 1]),
        ("total_population",   "Total population",   True,  None),
        ("p_clearance",        "p_clearance / day",  False, [0, 1]),
        ("n_strains",          "Active strains",      False, None),
    ]
    n_row, n_col = len(metric_info), len(SCENARIOS)

    subplot_titles = [
        f"{SCENARIO_NAMES.get(sc, sc)} – {ml}"
        for ml, _, _, _ in metric_info
        for sc in SCENARIOS
    ]
    fig = make_subplots(
        rows=n_row, cols=n_col,
        subplot_titles=subplot_titles,
        vertical_spacing=0.07,
        horizontal_spacing=0.08,
    )

    for ci, scenario in enumerate(SCENARIOS, start=1):
        sc_sub = sub[sub["scenario"] == scenario]
        for ri, (metric, _mlabel, log_y, ylim) in enumerate(metric_info, start=1):
            show_legend = ri == 1 and ci == 1
            for vi, (val, col) in enumerate(zip(values, colors)):
                vs = sc_sub[sc_sub["param_value"] == val]
                if vs.empty:
                    continue
                mean = vs.groupby("day")[metric].mean().reindex(days)
                std  = vs.groupby("day")[metric].std().reindex(days).fillna(0)
                upper = (mean + std).clip(lower=0).tolist()
                lower = (mean - std).clip(lower=0).tolist()

                fig.add_trace(go.Scatter(
                    x=days + list(reversed(days)),
                    y=upper + list(reversed(lower)),
                    fill="toself",
                    fillcolor=_rgba(col, 0.15),
                    line=dict(width=0),
                    showlegend=False,
                    hoverinfo="skip",
                ), row=ri, col=ci)

                fig.add_trace(go.Scatter(
                    x=days,
                    y=mean.tolist(),
                    mode="lines",
                    name=f"{val:.3g}",
                    legendgroup=f"v{vi}",
                    showlegend=show_legend,
                    line=dict(color=col, width=2),
                    hovertemplate=f"Tag %{{x}}: {val:.3g} → %{{y:.4f}}<extra></extra>",
                ), row=ri, col=ci)

            if ylim:
                fig.update_yaxes(range=ylim, row=ri, col=ci)
            if log_y:
                fig.update_yaxes(type="log", row=ri, col=ci)

    fig.update_layout(
        title=f"Trajektorien: {title}",
        height=270 * n_row + 80,
        width=max(900, 450 * n_col + 100),
        legend_title=title,
    )
    return fig


trajectory_fig(SELECTED_PARAM).show()

## 4 · Wert × Zeit-Heatmap

`resistant_fraction` als Funktion von **Parameterwert (y-Achse)** und **Tag (x-Achse)**.
Zeigt, ab welchem Tag ein hoher oder niedriger Parameterwert zu dauerhafter Persistenz oder schnellem Washout führt.
Beide Szenarien nebeneinander.

In [ ]:
def value_day_heatmap(param_field: str, metric: str = "resistant_fraction") -> go.Figure:
    sub    = traj[traj["param"] == param_field].copy()
    values = sorted(sub["param_value"].unique())
    days   = sorted(sub["day"].unique())
    title  = lbl(param_field)

    fig = make_subplots(
        rows=1, cols=len(SCENARIOS),
        subplot_titles=[SCENARIO_NAMES.get(sc, sc) for sc in SCENARIOS],
        horizontal_spacing=0.1,
    )
    zmax = max(
        1e-9,
        float(sub.groupby(["scenario", "day", "param_value"])[metric].mean().max()),
    )
    y_labels = [f"{v:.3g}" for v in values]

    for ci, scenario in enumerate(SCENARIOS, start=1):
        sc_sub = sub[sub["scenario"] == scenario]
        z = np.zeros((len(values), len(days)))
        for ri, val in enumerate(values):
            mean = (
                sc_sub[sc_sub["param_value"] == val]
                .groupby("day")[metric].mean()
                .reindex(days)
                .fillna(0)
            )
            z[ri] = mean.values

        fig.add_trace(
            go.Heatmap(
                z=z, x=days, y=y_labels,
                coloraxis="coloraxis",
                hovertemplate=(
                    f"Tag %{{x}}<br>{title}=%{{y}}<br>"
                    f"{METRIC_LABELS.get(metric, metric)}=%{{z:.3f}}<extra></extra>"
                ),
            ),
            row=1, col=ci,
        )

    fig.update_layout(
        title=f"{METRIC_LABELS.get(metric, metric)} – {title}",
        coloraxis=dict(
            colorscale="YlOrRd",
            cmin=0, cmax=zmax,
            colorbar=dict(title=METRIC_LABELS.get(metric, metric)),
        ),
        height=max(350, len(values) * 50 + 160),
        width=max(900, 460 * len(SCENARIOS) + 120),
        xaxis_title="Tag",
        yaxis_title=title,
    )
    return fig


value_day_heatmap(SELECTED_PARAM).show()

## 5 · Szenarien-Vergleich (kein ABX vs. beta-Lactam)

`resistant_fraction` am letzten Tag: **kein ABX (x)** vs. **beta-Lactam (y)**.
Jeder Punkt = ein Sweep-Wert eines Parameters; Farbe = Parameter.

- **Punkte oberhalb der Diagonalen** → beta-Lactam selektiert Resistenz stärker
- **Punkte unterhalb** → beta-Lactam eliminiert Resistenz
- **Auf der Diagonalen** → ABX-Druck ändert nichts

In [ ]:
_rf_end = summary[summary["metric"] == "resistant_fraction"].copy()
_rf_end["label"] = _rf_end["param"].map(lbl)

if {"no_abx", "beta_lactam"}.issubset(set(SCENARIOS)):
    _no  = _rf_end[_rf_end["scenario"] == "no_abx"][["param", "label", "param_value", "mean"]].rename(columns={"mean": "no_abx"})
    _abx = _rf_end[_rf_end["scenario"] == "beta_lactam"][["param", "param_value", "mean"]].rename(columns={"mean": "beta_lactam"})
    _cmp = _no.merge(_abx, on=["param", "param_value"])

    fig = px.scatter(
        _cmp, x="no_abx", y="beta_lactam",
        color="label",
        hover_data={"param_value": ":.3g", "label": True, "no_abx": ":.3f", "beta_lactam": ":.3f"},
        labels={
            "no_abx":       "resistant_fraction – kein ABX (Endtag)",
            "beta_lactam":  "resistant_fraction – beta-Lactam (Endtag)",
            "label":        "Parameter",
        },
        title="Szenarien-Vergleich: kein ABX vs. beta-Lactam",
    )
    fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1,
                  line=dict(dash="dot", color="gray", width=1))
    fig.update_layout(xaxis_range=[0, 1], yaxis_range=[0, 1], width=700, height=620)
    fig.show()
else:
    print("Beide Szenarien (no_abx und beta_lactam) werden für diesen Plot benötigt.")

## 6 · Finale-Tag Verteilung (Box-Plots)

Verteilung der einzelnen Seeds am letzten Tag für den gewählten Parameter.
Zeigt die stochastische Variabilität: bei kleinem N_SEEDS sollten Boxplots breiter sein.

In [ ]:
def final_day_boxplot(param_field: str, metric: str = "resistant_fraction") -> go.Figure:
    sub  = traj[(traj["param"] == param_field) & (traj["day"] == N_DAYS)].copy()
    sub["value_str"] = sub["param_value"].map(lambda v: f"{v:.3g}")
    values_sorted = [f"{v:.3g}" for v in sorted(sub["param_value"].unique())]
    sub["value_str"] = pd.Categorical(sub["value_str"], values_sorted, ordered=True)
    title = lbl(param_field)

    fig = px.box(
        sub, x="value_str", y=metric, color="scenario",
        color_discrete_map=SCENARIO_COLORS,
        points="all",
        category_orders={"value_str": values_sorted},
        labels={
            "value_str": title,
            metric:      METRIC_LABELS.get(metric, metric),
            "scenario":  "Szenario",
        },
        title=f"Verteilung am Tag {N_DAYS}: {title}",
    )
    if metric == "resistant_fraction":
        fig.update_yaxes(range=[0, 1])
    fig.update_layout(width=max(600, len(values_sorted) * 90 + 200), height=450)
    return fig


final_day_boxplot(SELECTED_PARAM).show()